# Session 5: Mini-Project -- Linear Regression End-to-End (2 hours)

**Goal:** Bring everything together into a complete, working linear regression implementation.

| Section | Time | Focus |
|---------|------|-------|
| Task 5.1 | 1 hr | Complete `LinearRegressionScratch` and fit data |
| Task 5.2 | 30 min | Experiments & analysis |
| Task 5.3 | 30 min | Documentation & reflection |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

%matplotlib inline

sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

---
## Task 5.1: Complete the Implementation (1 hour)

Open `linear_regression_from_scratch.py` and fill in all the TODO sections:
1. `_compute_cost` -- MSE calculation
2. `_compute_gradients` -- dW and db
3. `fit` -- training loop
4. `predict` -- forward pass
5. `r_squared` -- R^2 metric

Then come back here to test it.

<details>
<summary>💡 Click to see solution (linear_regression_from_scratch.py)</summary>

```python
# _compute_cost:
cost = np.mean((y_pred - y) ** 2) / 2

# _compute_gradients:
dw = X.T @ error / n_samples
db = np.mean(error)

# fit loop:
dw, db = self._compute_gradients(X, y)
self.weights -= self.learning_rate * dw
self.bias -= self.learning_rate * db
cost = self._compute_cost(X, y)
self.loss_history.append(cost)

# predict:
return X @ self.weights + self.bias

# r_squared:
ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - np.mean(y)) ** 2)
return 1 - ss_res / ss_tot
```
</details>


In [ ]:
from linear_regression_from_scratch import LinearRegressionScratch

### Generate Synthetic Data

In [ ]:
np.random.seed(42)

n_samples = 200
n_features = 1

# True relationship: y = 4.5 * x + 2.3 + noise
X = 2 * np.random.rand(n_samples, n_features)
y = 4.5 * X.squeeze() + 2.3 + np.random.randn(n_samples) * 0.8

# Train/test split (80/20)
split = int(0.8 * n_samples)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Training set: X={X_train.shape}, y={y_train.shape}")
print(f"Test set:     X={X_test.shape}, y={y_test.shape}")

plt.scatter(X_train, y_train, alpha=0.5, label='Train')
plt.scatter(X_test, y_test, alpha=0.5, label='Test')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Synthetic Data')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Train the Model

In [ ]:
model = LinearRegressionScratch(learning_rate=0.1, n_iterations=1000)
model.fit(X_train, y_train)

print(f"Learned weights: {model.weights}")
print(f"Learned bias: {model.bias:.4f}")
print(f"True weights: [4.5]")
print(f"True bias: 2.3")

### Plot Predictions and Loss Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

axes[0].scatter(X_train, y_train, alpha=0.4, label='Train data')
axes[0].scatter(X_test, y_test, alpha=0.4, label='Test data')
x_line = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
axes[0].plot(x_line, model.predict(x_line), 'r-', linewidth=2, label='Prediction')
axes[0].set_xlabel('X')
axes[0].set_ylabel('y')
axes[0].set_title('Linear Regression Fit')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: loss curve
axes[1].plot(model.loss_history)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Cost (MSE)')
axes[1].set_title('Training Loss')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('images/loss_history.png', dpi=150, bbox_inches='tight')
plt.show()

# R-squared
r2_train = model.r_squared(X_train, y_train)
r2_test = model.r_squared(X_test, y_test)
print(f"\nR² (train): {r2_train:.4f}")
print(f"R² (test):  {r2_test:.4f}")

---
## Task 5.2: Experiments & Analysis (30 min)

### Experiment 1: Learning Rate Sensitivity

In [ ]:
learning_rates = [0.001, 0.01, 0.1, 0.5, 1.0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lr in learning_rates:
    m = LinearRegressionScratch(learning_rate=lr, n_iterations=200)
    m.fit(X_train, y_train)
    final_loss = m.loss_history[-1] if m.loss_history else float('inf')
    r2 = m.r_squared(X_test, y_test)
    print(f"LR={lr:<5}: Final Loss={final_loss:.4f}, R²={r2:.4f}")

    axes[0].plot(m.loss_history, label=f'lr={lr}')

axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Cost')
axes[0].set_title('Loss Curves by Learning Rate')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, max(10, min(50, axes[0].get_ylim()[1])))

axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Cost (log scale)')
axes[1].set_title('Loss Curves (Log Scale)')
for lr in learning_rates:
    m = LinearRegressionScratch(learning_rate=lr, n_iterations=200)
    m.fit(X_train, y_train)
    axes[1].semilogy(m.loss_history, label=f'lr={lr}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Experiment 2: Compare with sklearn

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# sklearn model
sklearn_model = LinearRegression()
sklearn_model.fit(X_train, y_train)

# Our model (well-tuned)
our_model = LinearRegressionScratch(learning_rate=0.1, n_iterations=1000)
our_model.fit(X_train, y_train)

print("=== Comparison ===")
print(f"{'':20s} {'Ours':>12s} {'sklearn':>12s}")
print(f"{'Weight':20s} {our_model.weights[0]:12.4f} {sklearn_model.coef_[0]:12.4f}")
print(f"{'Bias':20s} {our_model.bias:12.4f} {sklearn_model.intercept_:12.4f}")

y_pred_ours = our_model.predict(X_test)
y_pred_sklearn = sklearn_model.predict(X_test)

print(f"{'R² (test)':20s} {r2_score(y_test, y_pred_ours):12.4f} {r2_score(y_test, y_pred_sklearn):12.4f}")

### Experiment 3: Multi-Feature Regression

In [ ]:
# Generate multi-feature data: y = 2*x1 + 3*x2 - 1*x3 + 5 + noise
np.random.seed(42)
n = 300
X_multi = np.random.randn(n, 3)
true_weights = np.array([2.0, 3.0, -1.0])
true_bias = 5.0
y_multi = X_multi @ true_weights + true_bias + np.random.randn(n) * 0.5

# Split
X_m_train, X_m_test = X_multi[:240], X_multi[240:]
y_m_train, y_m_test = y_multi[:240], y_multi[240:]

# Fit
model_multi = LinearRegressionScratch(learning_rate=0.01, n_iterations=2000)
model_multi.fit(X_m_train, y_m_train)

print(f"True weights:    {true_weights}")
print(f"Learned weights: {model_multi.weights}")
print(f"True bias:       {true_bias}")
print(f"Learned bias:    {model_multi.bias:.4f}")
print(f"R² (test):       {model_multi.r_squared(X_m_test, y_m_test):.4f}")

---
## Task 5.3: Documentation & Reflection (30 min)

### Week 1 Completion Checklist

- [ ] NumPy fundamentals (25+ exercises)
- [ ] Broadcasting visualization
- [ ] Matrix operations from scratch (3+ implemented)
- [ ] Gradient descent implementation
- [ ] Linear regression end-to-end
- [ ] R-squared > 0.8 on synthetic data

### Key Insights

**What I learned about vectorization:**

_Your answer..._

**What I learned about gradient descent:**

_Your answer..._

**What surprised me most:**

_Your answer..._

**What still feels fuzzy:**

_Your answer..._

### Final Checkpoint

Can you implement linear regression from scratch **without looking at your code**?

Try it in the cell below -- write the entire `fit` logic from memory.

In [ ]:
# Challenge: Implement linear regression fit from memory
# Given X (n_samples, n_features) and y (n_samples,),
# learn weights and bias using gradient descent.

def linear_regression_from_memory(X, y, lr=0.01, n_iters=1000):
    # YOUR CODE HERE (from memory!)
    pass


# Test it
# w, b = linear_regression_from_memory(X_train, y_train)
# print(f"Weights: {w}, Bias: {b:.4f}")

<details>
<summary>💡 Click to see solution</summary>

```python
def linear_regression_from_memory(X, y, lr=0.01, n_iters=1000):
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    for _ in range(n_iters):
        y_pred = X @ w + b
        dw = X.T @ (y_pred - y) / n
        db = np.mean(y_pred - y)
        w -= lr * dw
        b -= lr * db
    return w, b
```
</details>


---
### Before Week 2

Report back with:
1. Your GitHub repo link
2. One thing that surprised you
3. One concept still fuzzy
4. Your linear regression R^2 score